# Environment and data check

This notebook verifies the project environment and inspects PlantSeg as the primary dataset, PlantVillage as the secondary dataset, and PlantDoc as backup data. Use the **Python (plant-disease-assistant)** kernel.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from plant_disease.paths import OUTPUTS_DIR, SAMPLE_DIR

device = ('cuda' if torch.cuda.is_available() else
          'mps' if torch.backends.mps.is_available() else 'cpu')
print('Project:', PROJECT_ROOT)
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Device:', device)

## Sample manifest

The bundled manifest contains lightweight PlantVillage and PlantDoc examples. PlantSeg is the primary dataset and its exported image-mask example is shown below. Run `make init` to prepare and verify all full datasets.

## PlantSeg primary image and lesion mask

In [ ]:
example_dir = OUTPUTS_DIR / 'dataset_examples'
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(Image.open(example_dir / 'plantseg_example.jpg').convert('RGB'))
axes[0].set_title('PlantSeg field image')
axes[1].imshow(Image.open(example_dir / 'plantseg_example_mask.png'))
axes[1].set_title('Authoritative binary lesion mask')
for axis in axes:
    axis.axis('off')
plt.tight_layout()

In [ ]:
manifest = pd.read_csv(SAMPLE_DIR / 'manifest.csv')
display(manifest.groupby('dataset').size().rename('images'))
display(manifest.head())

## PlantVillage secondary controlled examples

In [ ]:
pv_rows = manifest[manifest.dataset == 'PlantVillage'].groupby('labels').head(1)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for axis, (_, row) in zip(axes.flat, pv_rows.iterrows()):
    image = Image.open(SAMPLE_DIR / row.file).convert('RGB')
    axis.imshow(image)
    axis.set_title(row.labels.replace('___', '\n'), fontsize=9)
    axis.axis('off')
plt.tight_layout()

## PlantDoc backup annotations

PlantDoc is retained as backup external or supplementary data. It stores bounding boxes as `[x, y, width, height]`; these mixed leaf/symptom annotations require cleaning before use.

In [ ]:
records = json.loads((SAMPLE_DIR / 'PlantDoc' / 'annotations.json').read_text())
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for axis, record in zip(axes.flat, records[:6]):
    image = Image.open(SAMPLE_DIR / 'PlantDoc' / record['file']).convert('RGB')
    draw = ImageDraw.Draw(image)
    for annotation in record['annotations']:
        x, y, width, height = annotation['bbox_xywh']
        draw.rectangle((x, y, x + width, y + height), outline='red', width=3)
    axis.imshow(image)
    axis.set_title(record['annotations'][0]['label'], fontsize=9)
    axis.axis('off')
plt.tight_layout()

## Next steps

1. Download the complete datasets.
2. Preprocess PlantSeg using `Metadata.csv` and binary masks as authoritative sources.
3. Train the classification baselines on PlantSeg.
4. Derive YOLO boxes from PlantSeg masks and compare Grad-CAM attention with those masks.
5. Map a PlantVillage subset for secondary controlled evaluation.
6. Clean PlantDoc only if backup external validation is required.